CREAZIONE API FLASK PER SERVIRE MODELLO AI

Con Flask il modello è accessibile ad altri software, è accessibile, per esempio, a un'applicazione ERP.

Creare un API Flask significa rendere il modello utilizzabile da altri programmi tramite richieste HTTP. 
Un'applicazione ERP invia dati, il tuo servizio Python esegue il modello e restituisce una risposta JSON.

Flask gestisce richieste, URL e risposte. La tua pipeline continua a gestire Llama, quantizzazione, RAG e sentiment analysis.
Il modello AI si carica una sola volta e rimane in memoria. Ogni richiesta riutilzza quella stessa istanza.
Possibili operazioni:
POST/genera     genera_testo()          utilizzato per chiedere la generazione di testo
POST/rag        rispondi_rag()          risposta basata sui documenti
POST/serntiment analyze_sentiment_json  classificazione di una recensione
GET/salute      controllo del servizio  verifica che il server risponda

Per l'installazione:
    -m pip install flask waitress

Waitress esegue l'applicazione Flask, supporta Windows e usa un singolo processo con più thread. Il server integrato di Flask è destinato allo sviluppo, per un servizio stabile si usa un server dedicato.

Flask facilita l'integrazione con un ERP reale, ma non migliora la correttezza del modello. Un JSON valido può contenere una conclusione sbagliata.
Una buona prassi potrebbe essere:
- calcoli e regole in python o SQL
- LLM per interpretazione dei testi, estrazione e spiegazioni
- validazione dei risultati prima di creare o modificare documenti ERP
- codice trasferito dal notebook a moduli .py per un servizio continuativo.

Esempio

$epicodeBody = @{
    domanda = "Quanto fa 7 x 8? Rispondi solo con il numero."
} | ConvertTo-Json

Invoke-RestMethod `
    -Uri "http://127.0.0.1:5000/genera" `
    -Method Post `
    -ContentType "application/json; charset=utf-8" `
    -Body ([Text.Encoding]::UTF8.GetBytes($epicodeBody))

Infine, TextStreamer stampa nella console. Per inviare progressivamente la risposta al client serve anche uno streaming HTTP; il semplice streaming=True della tua funzione non lo realizza


Flask: il Micro-Framework per l'AI
Prototipazione rapida di servizi di inferenza.
Flask è un micro-framework Python estremamente popolare per la creazione di API grazie alla sua semplicità e flessibilità
In ambito AI, viene utilizzato per 'incapsulare' il modello caricato in memoria e permettere a client esterni di inviare dati ed ottenere predizioni via HTTP.
La rete neurale carica i suoi pesi in memoria una volta sola, e Flask rimane in ascolto pronto a passare i dati al modello ed a restituire le rispota via http.

Anatomia di un Endpoint Predict
Dall'ascolto alla risposta.
- Routing: utilizzo del decoratore '@app.route' per mappare l'URL '/predict' alla funzione di inferenza python
- Metodo POST: poichè l'invio di dati al modello (testi, immagini o vettori) richiede un corpo nel messaggio, utilizzeremo esclusivamente il metodo POST. In POST  posso mettere tanti dati senza preoccuparmi della dimensione.
- Il request object: l'oggetto globale di Flask che permette di accedere ai dati JSON inviati dal client tramite 'request.get_json()'.
- L'endpoint funge da interfaccia tra il protocollo web e la logica matematica del modello sottostante.

Gestione delle Risposte
- jsonfy(): funziona di Flask, traduttore simultaneo che converte i dizionari python in stringhe JSON formattate correttamente con i relativi header HTTP
- status codes:importanza di restiture 200 per successo e codici 4xx/5xx in caso di dati malformati o errori interni del modello.
- model loading: il modello deve essere caricato una sola volta all'avvio del server e non dentro la funzione predict per evitare latenza inaccettabili

Ma quanto tempo ci mette una richiesta ad andare a tornare.
Il Ciclo di Vita della Richiesta
Dalla rete alla CPU
Ogni chiamata API attraversa uno strato di serializzazione che trasforma i pacchetti TCP in oggetti Python manipolabili
La latenza che l'utente percepisce è una somma di 3 fattori.
tempo toale = tempo network (tempo che impiega per viaggiare sui cavi di rete) + tempo parsing (tempo python) + tempo inference (tempo modello)
L'efficienza di questo strato determina il throughput massimo del nostro sistema di produzione.
In un sistema mal progettato il tempo di parsing (serializzazione/deserializzazione) può diventare un collo di bottiglia che rallenta tutto il sistema.

Deserializzazione dei Dati
Trasformare il testo in tensori
I dati che viaggiano su HTTP sono solo sequenze di caratteri, puro testo. Per darli in pasto a PyTorch o TensorFlow, dobbiamo eseguire una conversione di tipo. La deserializzazione trasforma il testo in matrici manipolabili
Dobbiamo mappare liste JSON in array NumPy o tensori pronti per la forward pass del modello.

Pipeline di Trasformazione
Dal formato stringa alla matrice numerica
- JSON to List: conversione automatica operata da Flask che trasforma il doby della richiesta in strutture dati native Python
- Numpy Bridge: uso di numpy come ponte per creare un array ottimizzato. np.array() per convertire le liste in matrici ottimizzate per il calcolo scientifico.
- Reshaping: necessità di garantire che il tensore di input rispetti la input_shape richiesta dal modello (es. batch size, canali, dimensioni). Se si aspetta un batch di immagini e noi gli diamo un solo vettore, il sistema di rompe. dobbiamo quindi fare reshape.
- La corretta deserializzazione deve prevedere anche la normalizzazione dei dati per allinearli alla distribuzione usata nel training.

Ma cosa succede se il client ci invia dati incompleti o del tipo sbagliato.
Validazione dell'Input
Verificare che il JSON contenga tutte le chiavi necessarie prima di invocare il modello per prevenire KeyError a runtime (schema checking).
Validare l'input prima di passare i dati al modello.
Controllare che i valori numerici non siano inviati come stringhe per evitare fallimenti silenziosi nelle operazioni matriciali (data type enforcement).
Utilizzo di blocchi try-except per catturare eccezioni durante la conversione e restituire messaggi d'errore chiari al client.

C'è però un limite fisico alla velocità di questo blocco 
Efficienza della Serializzazione
Ottimizzare il passaggio dei dati.
Inviare grossi array come liste JSON può essere molto lento a causa del parsing testuale del parser di Python.
La dimensione cel payload cresce con il numero di elementi.

Il collo di Bottiglia: Il GIL
Concorrenza e Parallelismo in Flask
Flask gestisce più utenti simultaneamente e il GIL (Global Interpreter Lock) può limitare le performance AI
Se il modello sta facendo calcoli pesanti sulla CPU il GIL blocca tutti gli altri utenti che stanno cercando di connettersi, finchè il calcolo non è finito.
Esistono strategie per mitigare questo limite in contesti CPU-bound

Comprendere il GIL
Il guardiano della memoria di Python
Il GIL è un meccanismo che impedisce a più thread python di eseguire bytecode simultaneamente, per evitare conflitti.
Nelle applicazione AI questo è un problema 
- CPU-bound AI: se il modello esegue inferenza pesante sulla CPU, il GIL blocca le altre richieste finchè il calcolo non termina.
- IO-bound vs CPU-bound: Flask eccelle nel gestire molte attese di rete, ma soffre se i core devono processare matrici in modo concorrente nello stesso processo.
- I tempo di risposta per n richieste simultanee può degradare linearmente se non si utilizza un modello a processi separati.

Senza una stretegia corretta, il tempo di risposta totale cresce linearmente in base al numero di utenti.
Fortunamente abbiamo un modo per aggirare questo ostacolo. La soluzione di chiama multi-processing

Strategia di Scalabilità
- Multi-processing: avviare più istanze (worker) del server Flask per permettere ad ogni core della CPU di gestire un modello in modo indipendente.
Invece di avere un solo ufficio con un solo timbro, apriamo molti uffici ognuno con il suo timbro e il proprio modello caricato.
- WSGI Servers: uso di strumenti come Gunicorn per orchiestrare i processi worker e bypassare i limiti del server di sviluppo integrato in Flask. Ogni worker è un processo indipendente con il proprio spazio di memoria ed il proprio accesso alla GPU.
In questo modo se worker1 è impegnato a processare un'immagine dell'utente A, worker 2 può tranquillamente rispondere all'utente B su un altro processore.
- Threading Limitations: I thread in Flask sono utili per l'input/output ma non scalano per l'inferenza di modelli di Deep Learning pesanti a causa dei GIL.

Ma come decidiamo quanti di questi worker creare?
Verso il Deployment Reale
Oltre il server di sviluppo.
Non si deve mai usare app.run() (server di sviluppo che si lancia con Flask) in produzione poichè è single-process, non sicuro e serve per il test.
In produzione la soluzione standard prevede l'uso di un application server (server vsgai professionale) che gestisce un pool di processi worker paralleli.
La regola empirica classica è di impostare un numero di worker pari al doppio dei numero dei processori + 1
Workers = 2 * Cores + 1
Questo bilanciamento permete di saturare la macchina senza soffocarla.

In [6]:
import torch
import torch.nn as nn
import numpy as np
from flask import Flask, request, jsonify

# Definizione della struttura del modello neurale.
# Questa classe eredita da nn.Module, la base per tutti i moduli di rete neurale in PyTorch.
class HealthClassifier(nn.Module):
    def __init__(self):
        # Inizializzazione della classe base.
        super(HealthClassifier, self).__init__()
        # Definiamo uno strato lineare (fully connected).
        # Prende in input 4 caratteristiche e restituisce 2 valori di uscita (le classi).
        self.linear = nn.Linear(4, 2) 

    def forward(self, x):
        # Il metodo forward definisce come i dati fluiscono attraverso la rete.
        # Qui i dati passano semplicemente attraverso lo strato lineare.
        return self.linear(x)

# Preparazione dell'ambiente di esecuzione: usiamo la CPU per l'inferenza.
device = torch.device("cpu")

# Creazione di un'istanza del modello e spostamento sulla memoria del dispositivo (CPU).
model = HealthClassifier().to(device)

# Predisposizione del modello per la fase di valutazione.
# Questo passaggio è fondamentale perché disabilita comportamenti specifici dell'addestramento 
# come il Dropout o la Batch Normalization, garantendo risultati costanti.
model.eval()

# Inizializzazione dell'applicazione Flask per creare il server web.
app = Flask(__name__)

# Definizione dell'endpoint API che riceverà le richieste di classificazione.
# Accetta solo richieste di tipo POST.
@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Estrazione del corpo della richiesta in formato JSON.
        data = request.get_json()
        
        # Validazione minimale: verifichiamo che i dati esistano e contengano la chiave corretta.
        if not data or 'features' not in data:
            return jsonify({"error": "Dati mancanti. Inviare un JSON con la chiave 'features'"}), 400

        # Recupero della lista di caratteristiche inviate dall'utente.
        raw_features = data['features']
        
        # Conversione dei dati grezzi in un array NumPy con precisione a 32 bit (float32).
        # PyTorch lavora in modo ottimale con questa precisione numerica.
        input_array = np.array(raw_features, dtype=np.float32)
        
        # Conversione da array NumPy a tensore PyTorch e aggiunta di una dimensione.
        # Il modello si aspetta un input in formato (batch_size, num_features).
        # unsqueeze(0) trasforma un vettore (4) in una matrice (1, 4).
        input_tensor = torch.from_numpy(input_array).unsqueeze(0).to(device)

        # Fase di inferenza (calcolo del risultato).
        # Usiamo torch.no_grad() per evitare di calcolare i gradienti, riducendo l'uso di memoria e aumentando la velocità.
        with torch.no_grad():
            # Passiamo il tensore al modello (chiama internamente il metodo forward).
            logits = model(input_tensor)
            
            # Applichiamo la funzione Softmax per convertire i valori di uscita (logits) in probabilità.
            # dim=1 indica che la somma delle probabilità deve essere 1 per ogni riga del batch.
            probabilities = torch.softmax(logits, dim=1)
            
            # Determiniamo la classe con la probabilità più alta.
            # item() estrae il valore scalare dal tensore di output.
            prediction = torch.argmax(probabilities, dim=1).item()
            
            # Estraiamo il valore di confidenza associato alla previsione.
            confidenza = torch.max(probabilities).item()

        # Mappatura del risultato numerico in una stringa leggibile.
        # In questo scenario: 1 = In Salute, 0 = Affaticato.
        status = "In Salute" if prediction == 1 else "Affaticato"
        
        # Restituzione della risposta al client in formato JSON.
        return jsonify({
            "status": status,
            "confidence": f"{confidenza:.2%}",
            "model_version": "2026.1.0"
        }), 200

    except Exception as e:
        # Gestione generica degli errori per evitare il crash del server.
        # Restituisce il messaggio dell'errore e lo status code 500 (Internal Server Error).
        return jsonify({"error": str(e)}), 500

# Punto di ingresso dello script.
if __name__ == '__main__':
    # Avvio del server Flask in ascolto su tutte le interfacce di rete (0.0.0.0) sulla porta 5000.
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://120.0.0.108:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Oct/2026 19:36:04] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 19:36:10] "POST /predict HTTP/1.1" 500 -
